In [12]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/同花顺')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:6974,第一支股票:000001


,Id,Code,Date,Open,High,Low,Close,Amount,Volume
0,1,000001,2014-01-02 00:00:00,12.12,12.30,12.05,12.23,596223740.0,48991089.0
1,2,000001,2014-01-03 00:00:00,12.15,12.16,11.78,11.93,656631300.0,55111484.0
2,3,000001,2014-01-06 00:00:00,11.89,12.00,11.50,11.67,679280380.0,58211823.0
3,4,000001,2014-01-07 00:00:00,11.53,11.76,11.51,11.63,393977580.0,33840749.0
4,5,000001,2014-01-08 00:00:00,11.64,11.95,11.53,11.76,538436170.0,45776816.0


# 单个股票

In [13]:
dt['avgPrice'] = dt['Amount']/dt['Volume'] # 交易额除以交易次数等于平均价格，不同于收盘价
dt.head()

,Id,Code,Date,Open,High,Low,Close,Amount,Volume,avgPrice
0,1,000001,2014-01-02 00:00:00,12.12,12.30,12.05,12.23,596223740.0,48991089.0,12.170045
1,2,000001,2014-01-03 00:00:00,12.15,12.16,11.78,11.93,656631300.0,55111484.0,11.914600
2,3,000001,2014-01-06 00:00:00,11.89,12.00,11.50,11.67,679280380.0,58211823.0,11.669114
3,4,000001,2014-01-07 00:00:00,11.53,11.76,11.51,11.63,393977580.0,33840749.0,11.642106
4,5,000001,2014-01-08 00:00:00,11.64,11.95,11.53,11.76,538436170.0,45776816.0,11.762202


In [14]:
dt['nextClose']=dt['Close'].shift(-1)  # 明天的收盘价
dt['nextCloseRate']=(dt['nextClose']/dt['Close']-1)*100
dt.head()

,Id,Code,Date,Open,High,Low,Close,Amount,Volume,avgPrice,nextClose,nextCloseRate
0,1,000001,2014-01-02 00:00:00,12.12,12.30,12.05,12.23,596223740.0,48991089.0,12.170045,11.93,-2.452984
1,2,000001,2014-01-03 00:00:00,12.15,12.16,11.78,11.93,656631300.0,55111484.0,11.914600,11.67,-2.179380
2,3,000001,2014-01-06 00:00:00,11.89,12.00,11.50,11.67,679280380.0,58211823.0,11.669114,11.63,-0.342759
3,4,000001,2014-01-07 00:00:00,11.53,11.76,11.51,11.63,393977580.0,33840749.0,11.642106,11.76,1.117799
4,5,000001,2014-01-08 00:00:00,11.64,11.95,11.53,11.76,538436170.0,45776816.0,11.762202,11.82,0.510204


In [15]:
# 这里要计算一下某一列的平均值
mean1 = dt.loc[dt['avgPrice']>dt['Close'],:]['nextCloseRate'].mean()
mean2 = dt.loc[dt['avgPrice']==dt['Close'],:]['nextCloseRate'].mean()
mean3 = dt.loc[dt['avgPrice']<dt['Close'],:]['nextCloseRate'].mean()
print(f'平均价格大于收盘价:{mean1},平均价格等于收盘价:{mean2},平均价格小于收盘价:{mean3},')
# 可以看到如果平均价格大于收盘价，那么未来

平均价格大于收盘价:0.08919805484137862,平均价格等于收盘价:nan,平均价格小于收盘价:-0.052459310521052634,


# 所有股票

In [18]:
mean1s = []
mean2s = []
mean3s = []


for i in tqdm(range(len(codes))):
    code = codes[i]
    dt = datasource.getData(code)
    dt['avgPrice'] = dt['Amount']/dt['Volume'] # 交易额除以交易次数等于平均价格，不同于收盘价
    dt['nextClose']=dt['Close'].shift(-1)  # 明天的收盘价
    dt['nextCloseRate']=(dt['nextClose']/dt['Close']-1)*100
    dt = dt.dropna()
    mean1 = dt.loc[(dt['avgPrice']>dt['Close']) & (abs(dt['nextCloseRate']) < 10),:]['nextCloseRate'].mean()
    mean2 = dt.loc[(dt['avgPrice']==dt['Close']) & (abs(dt['nextCloseRate']) < 10),:]['nextCloseRate'].mean()
    mean3 = dt.loc[(dt['avgPrice']<dt['Close']) & (abs(dt['nextCloseRate']) < 10),:]['nextCloseRate'].mean()
    if not np.isnan(mean1) :
        mean1s.append(mean1)
    if not np.isnan(mean2) :
        mean2s.append(mean2)
    if not np.isnan(mean3) :
        mean3s.append(mean3)

# 这里计算平均值
if len(mean1s) >0:
    print(f'大于的情况:{sum(mean1s)/len(mean1s)},最大:{max(mean1s)}，最小:{min(mean1s)}')
if len(mean2s) >0:
    print(f'等与的情况:{sum(mean2s)/len(mean2s)},最大:{max(mean2s)}，最小:{min(mean2s)}')
if len(mean3s) >0:
    print(f'小于的情况:{sum(mean3s)/len(mean3s)},最大:{max(mean2s)}，最小:{min(mean3s)}')
    

100%|██████████████████████████████████████████████████████████████████████████████| 6974/6974 [01:45<00:00, 66.24it/s]

大于的情况:0.038077507095222236,最大:9.568331012532315，最小:-4.190169218372275
等与的情况:-1.015248818592238,最大:9.999999999999986，最小:-9.999999999999998
小于的情况:0.026208952746478505,最大:9.999999999999986，最小:-8.18181818181818


看起来难分彼此啊。